# 🚨 Delincuencia en España
# ¿Quién delinque más
# en términos relativos?

### 💼 PREGUNTA ENTREVISTA:
### Cruza delitos y población
### del INE y calcula la tasa
### por cada 1.000 habitantes.

**Truco:** en cifras absolutas
gana quien más gente tiene.
Lo interesante es la tasa.

#### 1️⃣ Cargamos los delitos

In [ ]:
import pandas as pd

delitos = pd.read_csv(
  '../datasets/'
  'delicuencia_nacionalidad.csv',
  sep=';',
  dtype={'Total': str}
)

# El INE usa el '.' como
# separador de miles: lo quitamos
delitos['Total'] = pd.to_numeric(
  delitos['Total']
  .str.replace('.', '',
    regex=False),
  errors='coerce'
)

delitos.head()

#### 2️⃣ Nos quedamos con el TOTAL

Los datos son jerárquicos.
La fila con 'Nivel 2' vacío
es el total (sin desglose).

In [ ]:
es_total = (
  delitos['Tipo de Delito: Nivel 2']
  .isna()
)
es_2024 = delitos['Periodo'] == 2024

delitos = (
  delitos[es_total & es_2024]
  .set_index('Nacionalidad')
  ['Total']
)

delitos

#### 3️⃣ Cargamos la población

Para la tasa necesitamos
cuánta gente hay de cada origen.

In [ ]:
poblacion = pd.read_csv(
  '../datasets/'
  'evolucion_migracion.csv',
  sep=';'
)

poblacion['Total'] = pd.to_numeric(
  poblacion['Total']
  .str.replace('.', '',
    regex=False),
  errors='coerce'
)

# Total nacional, enero 2024
filtro = (
  (poblacion['Sexo'] == 'Total')
  & (poblacion['Periodo']
     == '1 de enero de 2024')
  & (poblacion[
      'Grupo quinquenal de edad']
     == 'Todas las edades')
)

poblacion = (
  poblacion[filtro]
  .set_index('Nacionalidad')
  ['Total']
)

#### 4️⃣ Agrupamos por continente

Los dos datasets no usan
las mismas categorías,
así que las unificamos.

In [ ]:
def tasa(delitos_grupo,
         poblacion_grupo):
  """Delitos por 1.000 hab."""
  return (
    delitos_grupo
    / poblacion_grupo
    * 1000
  )

# América = Norte+Centro+Sur
pob_america = (
  poblacion['De América del Norte']
  + poblacion[
      'De Centro América y Caribe']
  + poblacion['De Sudamérica']
)

# Europa = UE + resto de Europa
ue = 'País de la UE27_2020 sin España'
resto = 'País de Europa menos UE27_2020'

del_europa = (
  delitos[ue] + delitos[resto]
)
pob_europa = (
  poblacion[ue] + poblacion[resto]
)

#### 5️⃣ Calculamos la tasa 🥁

In [ ]:
ranking = pd.Series({
  'Española': tasa(
    delitos['Española'],
    poblacion['Española']),
  'Europa': tasa(
    del_europa, pob_europa),
  'África': tasa(
    delitos['De Africa'],
    poblacion['De Africa']),
  'América': tasa(
    delitos['De América'],
    pob_america),
  'Asia': tasa(
    delitos['De Asia'],
    poblacion['De Asia']),
})

ranking = ranking.sort_values(
  ascending=False
)

print("Delitos x 1.000 hab (2024)")
print("-" * 28)
print(ranking.round(1))

veces = (
  ranking.iloc[0]
  / ranking['Española']
)
print("-" * 28)
print(f"El grupo más alto delinque")
print(f"{veces:.1f}x más que la")
print(f"población española.")